# Figures for the DH 2027 poster

Based on `resources/final/` (output of `gnd_pipeline.ipynb`). Two figures, revised from `analysis_v2.ipynb` (old repo):

| Fig. | Question | Change compared to `analysis_v2.ipynb` |
|---|---|---|
| 1 | How are the **works** of the corpora distributed over the rank classes? | works instead of authors |
| 14 | Size, period, source: three factors at a glance | genre dropped; panels stacked, larger; works instead of authors |
| 15 | Gender: share of works by non-male authors vs. canonicity | new |

**Works and canonicity.** `data_all.csv` has one row per work. A work with several authors gets the rank of its **most canonical author**
(`is_primary` in `gnd_pipeline.ipynb`, step 4). Works without a GND ID (anonymous, placeholders) count as *no rank*.

**Corpus selection** as in `analysis_v2.ipynb`: without translated fiction, de-Corp non-fiction and DisKo classics;
Bildungsromankorpus only works with full text; 19C prose one row per work (the corpus lists episodes).

Figures are saved to `figures/` as PNG (needs `kaleido`); with `SAVE_HTML = True` also as HTML (hover).

# Settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"   # plotly.js from the CDN, keeps the notebook small

RES = Path("../resources")
FINAL = RES / "final"
DATA_FILE = FINAL / "data_all.csv"                  # one row per work
AUTHORS_FILE = FINAL / "work_authors.csv"           # one row per (work, author)
GND_FILES = [RES / "gnd_all_mit_namensvarianten.csv", RES / "gnd_add.csv"]   # gender of the authors (Fig. 15)
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

TOP = 200                        # "canonical" = rank 1..TOP
UNIT14 = "works"                 # Fig. 14 and 15: "works" (as Fig. 1) or "authors" (as in analysis_v2)
COLOR14 = "uniform"              # Fig. 14 dot colour: "uniform" (one colour, DOT_COLOR) or a measure on the purple ramp:
                                 # "score" (median canonicity score), "listed" (share in the canon list), "median_rank"
DOT_COLOR = "#1baf7a"            # Fig. 14, COLOR14 = "uniform"
BILDUNG_FULLTEXT_ONLY = True     # Bildungsromankorpus: drop titles without full text
SAVE_PNG = True
SAVE_HTML = False                # also save interactive HTML (hover) next to the PNG
SHOW_TITLES = False              # figure titles (off for the poster: captions go next to the figure)

NAMES = {                        # order = order of the corpus table: alphabetical by full name, 19C prose last
    "07_bildungsromane": "Bildungsromane",
    "01_german-fiction": "CGLF",
    "04_d-prose": "d-prose",
    "06_romane-der-romantik": "d-RoRo",
    "03_de-corp": "de-Corp",
    "13_deutscher-novellenschatz": "Novellenschatz",
    "11_dta-belletristik": "DTA fiction",
    "09_ecocor": "EcoCor-DE",
    "05_ELTeC-deu": "ELTeC-deu",
    "08_glance": "glance",
    "14_neuer-deutscher-novellenschatz": "Neuer Novellenschatz",
    "10_19C-prose": "19C prose",
}
LABELS = {"Bildungsromane": "Bildungsromankorpus", "CGLF": "Corpus of German Language Fiction",
          "Novellenschatz": "Deutscher Novellenschatz", "DTA fiction": "DTA core corpus (fiction)",
          "Neuer Novellenschatz": "Neuer Deutscher Novellenschatz", "19C prose": "19CproseCorpus"}   # as in the corpus table

## Colours and layout

In [2]:
# rank classes: one hue (purple), dark = more canonical; "no rank" neutral grey
CLASSES = ["1–50", "51–200", "201–500", "501–1000", "> 1000", "no rank"]
CLASS_COLORS = ["#3f007d", "#5e3c99", "#8073ac", "#a99dcd", "#d3cce8", "#e3e2dd"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e8e7e3", "#ffffff"

def style(fig, title, subtitle=None, height=480, legend=True, font_size=13):
    t = f"<b>{title}</b>" + (f"<br><span style='font-size:13px;color:{INK2}'>{subtitle}</span>" if subtitle else "")
    fig.update_layout(
        title=dict(text=t if SHOW_TITLES else "", x=0, xanchor="left", y=1, yanchor="top", yref="container", pad=dict(t=16),
                   font=dict(size=font_size + 4, color=INK)),
        template="simple_white", height=height, paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        font=dict(family="Helvetica, Arial, sans-serif", size=font_size, color=INK2),
        margin=dict(t=(90 if SHOW_TITLES else 45) + (40 if legend else 0), l=10, r=20, b=40), showlegend=legend,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, title_text=""),
        hoverlabel=dict(bgcolor="white", font_size=13),
    )
    fig.update_xaxes(showgrid=False, linecolor=GRID, ticks="", title_font=dict(color=INK2))
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, ticks="", title_font=dict(color=INK2))
    return fig

def show(fig, name, post_script=None):
    if SAVE_PNG:
        fig.write_image(FIG_DIR / f"{name}.png", scale=2, width=fig.layout.width or 1000, height=fig.layout.height)
    if SAVE_HTML:
        fig.write_html(FIG_DIR / f"{name}.html", include_plotlyjs="cdn", post_script=post_script)
    fig.show()

# Load data

In [3]:
data = pd.read_csv(DATA_FILE, dtype={"gndIdentifier": str}, low_memory=False)
data = data[data["corpus"].isin(NAMES)].copy()     # without translated fiction, de-Corp non-fiction, DisKo classics
if BILDUNG_FULLTEXT_ONLY:
    data = data[~((data["corpus"] == "07_bildungsromane") & data["filename"].isna())]
# 19C prose: one row per episode -> one row per work
is19 = data["corpus"] == "10_19C-prose"
data = pd.concat([data[~is19], data[is19].drop_duplicates(subset=["author_first", "author_last", "title"])])
# check: the d-prose years come from its metadata (norm_year); a pipeline export without them drops d-prose from Fig. 14
for corpus, d_ in data.groupby("corpus"):
    if d_["year"].isna().all():
        print(f"WARNING: no years for {corpus}")
data["name"] = data["corpus"].map(NAMES)
data["cls"] = (pd.cut(data["canonicity_rank"], [0, 50, 200, 500, 1000, np.inf], labels=CLASSES[:-1])
               .astype(object).fillna("no rank"))

# all authors of the kept works (co-authors included), one row per author and corpus
wa = pd.read_csv(AUTHORS_FILE, dtype={"gndIdentifier": str}, low_memory=False)
wa = wa[wa["work_id"].isin(data["work_id"]) & wa["gndIdentifier"].ne("NoID")]
authors = (wa.assign(name=wa["corpus"].map(NAMES))
           .groupby(["name", "gndIdentifier"], as_index=False)
           .agg(works=("work_id", "size"), rank=("canonicity_rank", "first"), score=("canonicity_score", "first"), author=("GND_name", "first")))

ORDER = list(NAMES.values())
n_w, n_a = data.groupby("name").size(), authors.groupby("name").size()
overview = pd.DataFrame({"works": n_w, "authors": n_a,
                         "works: top200": data.groupby("name")["canonicity_rank"].apply(lambda r: (r <= TOP).mean()),
                         "authors: top200": authors.groupby("name")["rank"].apply(lambda r: (r <= TOP).mean())}).loc[ORDER]
overview.style.format({"works": "{:,.0f}", "authors": "{:,.0f}", "works: top200": "{:.0%}", "authors: top200": "{:.0%}"})

,works,authors,works: top200,authors: top200
name,,,,
Bildungsromane,126,73,47%,36%
CGLF,"2,735",547,29%,12%
d-prose,"2,511",332,24%,10%
d-RoRo,115,41,46%,41%
de-Corp,"4,215","1,009",20%,8%
Novellenschatz,87,82,24%,22%
DTA fiction,553,239,43%,31%
EcoCor-DE,190,89,32%,27%
ELTeC-deu,100,56,44%,32%


# Fig. 1: Rank classes of the works

The **works** of each corpus by rank class of their author (the canonicity scores are author-based; with several authors the most canonical one counts).
One hue from dark (canonical) to light, grey = author not in the canon list or no GND ID.
Sorted by the share of works in the top 200 (rank 1–200). Below each name: number of works and authors.

Compared to the author-based version, prolific authors now weigh by their number of works (e.g. Karl May in d-prose).

In [4]:
w_cnt = data.groupby(["name", "cls"]).size().unstack(fill_value=0).reindex(columns=CLASSES, fill_value=0)
w_sh = w_cnt.div(w_cnt.sum(axis=1), axis=0)
order1 = (w_sh["1–50"] + w_sh["51–200"]).sort_values().index.tolist()   # most canonical on top
ylab = [f"<span style='color:{INK}'><b>{LABELS.get(n, n)}</b></span><br>"
        f"<span style='font-size:11px;color:{INK2}'>{n_w[n]:,} works · {n_a[n]:,} authors</span>" for n in order1]

fig = go.Figure()
for cls, colr in zip(CLASSES, CLASS_COLORS):
    fig.add_bar(y=ylab, x=w_sh.loc[order1, cls], orientation="h", name=cls,
                legendgroup="unranked" if cls == "no rank" else "ranked",
                marker=dict(color=colr, line=dict(color=SURFACE, width=1.5)),
                customdata=np.c_[[LABELS.get(n, n) for n in order1], w_cnt.loc[order1, cls], n_w[order1]],
                hovertemplate="<b>%{customdata[0]}</b><br>" + ("no rank" if cls == "no rank" else "rank " + cls)
                              + ": %{customdata[1]} of %{customdata[2]} works (%{x:.1%})<extra></extra>")
fig.update_layout(barmode="stack", bargap=0.3)
fig.update_xaxes(tickformat=".0%", range=[0, 1], tickvals=[0, 0.5, 1], title="share of works")
style(fig, "How canonical are the works of the corpora?", "Rank of the author in the canon list (1 = Goethe)", height=680)
fig.update_layout(width=1000, margin=dict(t=(100 if SHOW_TITLES else 25), r=20, b=50),
                  legend=dict(orientation="v", x=1.02, xanchor="left", y=1, yanchor="top", traceorder="grouped",
                              tracegroupgap=14, title=dict(text="Rank of author<br>in canon list", font=dict(size=13, color=INK))))
show(fig, "fig01_rank_classes_works")
w_sh.loc[order1[::-1]].style.format("{:.0%}")

cls,1–50,51–200,201–500,501–1000,> 1000,no rank
name,,,,,,
Bildungsromane,40%,7%,16%,12%,18%,7%
d-RoRo,34%,12%,20%,17%,15%,2%
ELTeC-deu,25%,19%,24%,15%,17%,0%
DTA fiction,26%,17%,22%,13%,11%,11%
EcoCor-DE,22%,9%,23%,12%,26%,8%
CGLF,17%,12%,15%,16%,28%,11%
19C prose,17%,8%,7%,8%,24%,36%
Novellenschatz,15%,9%,11%,9%,32%,23%
d-prose,8%,16%,19%,17%,29%,12%


# Fig. 14: Three factors at a glance

Same y-axis in all panels: share of the authors (or works, see `UNIT14`) of a corpus in the top 200. Each dot is a corpus;
only the x-axis changes:
- **Size:** number of authors (or works), log scale.
- **Period:** median publication year of the works (Deutscher Novellenschatz: anthology years 1871–76, since `year` is the 1910 edition).
- **Source:** share of texts from Project Gutenberg (DE + US) among the texts with a known source;
  corpora without a source for most texts (EcoCor-DE, 19C prose, Neuer Novellenschatz) are left out.

Source per text (columns of the per-corpus files in `resources/final/`):
glance `fulltext_source`; de-Corp `source`; Bildungsromane URL in „Quelle des Volltextes“ (TextGrid handles → TextGrid);
d-RoRo `Repository`; d-prose `Repositorium`;
CGLF all Gutenberg-DE; DTA fiction and Novellenschatz all DTA; ELTeC-deu TextGrid or DTA (corpus README, no source per text, but none from Gutenberg).

**Colour:** one colour for all corpora (`COLOR14 = "uniform"`). Tried as colour measures (`COLOR14`): median canonicity score,
share of authors in the canon list, median rank. All three follow the y-axis closely, so they add little (comparison in `figures/fig14_colour_variants/`).

With 12 corpora the factors cannot be separated: the large corpora are also the late Gutenberg corpora.

In [5]:
import re

def url_source(u):
    if not isinstance(u, str): return None
    if "hdl.handle.net/11858" in u: return "TextGrid"
    if "deutschestextarchiv.de" in u: return "DTA"
    if "projekt-gutenberg.org" in u: return "Gutenberg-DE"
    if re.search(r"(^|//|\.)gutenberg\.org", u): return "Gutenberg-US"
    return "other"

PG = {"PG-DE": "Gutenberg-DE", "PG-US": "Gutenberg-US", "TextGrid": "TextGrid", "DTA": "DTA"}
src_map = pd.concat([
    pd.read_csv(FINAL / "08_glance.csv").assign(source=lambda x: x["fulltext_source"].map(PG)),
    pd.read_csv(FINAL / "03_de-corp.csv").assign(source=lambda x: x["source"].map(PG)),
    pd.read_csv(FINAL / "07_bildungsromane.csv").assign(source=lambda x: x["Quelle des Volltextes"].map(url_source)),
    pd.read_csv(FINAL / "06_romane-der-romantik.csv")
      .assign(source=lambda x: x["Repository"].map({"TextGrid": "TextGrid", "Projekt Gutenberg": "Gutenberg-DE"})),
    pd.read_csv(FINAL / "04_d-prose.csv")
      .assign(source=lambda x: x["Repositorium"].map({"gutenberg": "Gutenberg-DE", "textgrid": "TextGrid", "DTA": "DTA"})),
])[["work_id", "source"]].drop_duplicates("work_id")

src = data.merge(src_map, on="work_id", how="left", validate="1:1")
src.loc[src["corpus"].eq("01_german-fiction"), "source"] = "Gutenberg-DE"
src.loc[src["corpus"].isin(["11_dta-belletristik", "13_deutscher-novellenschatz"]), "source"] = "DTA"
src.loc[src["corpus"].eq("05_ELTeC-deu"), "source"] = "TextGrid/DTA"   # corpus README: "Sources: Textgrid Repository, Deutsches Textarchiv"; none from Gutenberg

known = src[src["source"].notna()]
gut = known.groupby("name")["source"].apply(lambda q: q.str.startswith("Gutenberg").mean())
coverage_src = (known.groupby("name").size() / n_w).reindex(gut.index)
gut = gut[coverage_src >= 0.5]   # only corpora with a source for most texts

if UNIT14 == "authors":
    top14 = authors.groupby("name")["rank"].apply(lambda r: (r <= TOP).mean())
    size14 = n_a
    score14 = authors.groupby("name")["score"].apply(lambda s: s.fillna(0).median())
    ranks14 = authors.groupby("name")["rank"]
else:
    top14 = data.groupby("name")["canonicity_rank"].apply(lambda r: (r <= TOP).mean())
    size14 = n_w
    score14 = data.groupby("name")["canonicity_score"].apply(lambda s: s.fillna(0).median())
    ranks14 = data.groupby("name")["canonicity_rank"]
fac = pd.DataFrame({"top": top14, "size": size14, "year": data.groupby("name")["year"].median(),
                    "gutenberg": gut, "score": score14,     # score: median canonicity score, not in canon list = 0
                    "listed": ranks14.apply(lambda r: r.notna().mean()),                  # share in the canon list
                    "median_rank": ranks14.apply(lambda r: r.fillna(np.inf).median()),    # not listed = below the list
                    }).loc[ORDER]
fac.loc["Novellenschatz", "year"] = 1873   # `year` is the digitised edition (mostly 1910); anthology years 1871–76
fac.style.format({"top": "{:.0%}", "size": "{:,.0f}", "year": "{:.0f}", "gutenberg": "{:.0%}", "score": "{:.2f}", "listed": "{:.0%}", "median_rank": "{:,.0f}"}, na_rep="–")

,top,size,year,gutenberg,score,listed,median_rank
name,,,,,,,
Bildungsromane,47%,126,1854,20%,9.41,93%,238
CGLF,29%,"2,735",1895,100%,2.38,89%,722
d-prose,24%,"2,511",1901,72%,2.35,88%,741
d-RoRo,46%,115,1812,24%,5.61,98%,339
de-Corp,20%,"4,215",1897,100%,2.05,80%,"1,433"
Novellenschatz,24%,87,1873,0%,2.05,77%,"1,433"
DTA fiction,43%,553,1800,0%,8.27,89%,264
EcoCor-DE,32%,190,1887,–,4.13,92%,410
ELTeC-deu,44%,100,1881,0%,8.50,100%,256


In [6]:
FS = 16                          # base font size (poster)
UNIT_LABEL = {"authors": "authors", "works": "works"}[UNIT14]
PANELS = [   # column, heading (bold part, explanation of the x-axis)
    ("size", "Size", f"number of {UNIT_LABEL}, log scale"),
    ("gutenberg", "Source", "share of texts from Project Gutenberg"),
    ("year", "Period", "median publication year"),
]
ROW = {xc: r for r, (xc, _, _) in enumerate(PANELS, start=1)}   # panel order: change PANELS only
POS14 = {   # label positions where the default ("middle right") would collide; depend on UNIT14
    "works": {
        "size": {"d-RoRo": "top left", "Bildungsromane": "top right", "ELTeC-deu": "middle left", "19C prose": "middle left",
                 "CGLF": "top center", "glance": "bottom center"},
        "gutenberg": {"ELTeC-deu": "top right", "DTA fiction": "bottom right", "Bildungsromane": "top left",
                      "CGLF": "top center", "de-Corp": "bottom center", "glance": "middle left"},
        "year": {"EcoCor-DE": "middle left", "19C prose": "middle left", "Neuer Novellenschatz": "bottom center"},
    },
    "authors": {
        "size": {"19C prose": "middle left", "CGLF": "top center", "glance": "bottom center"},
        "gutenberg": {"CGLF": "top center", "de-Corp": "bottom center", "glance": "middle left",
                      "ELTeC-deu": "top right", "DTA fiction": "bottom right"},
        "year": {"Neuer Novellenschatz": "top left", "19C prose": "middle left", "CGLF": "top center",
                 "glance": "middle left", "de-Corp": "bottom center"},
    },
}[UNIT14]
# dot colour on the purple ramp of Fig. 1 (dark = more canonical); see COLOR14 in the settings
DOT_SCALE = [[0, "#b3a9d6"], [0.5, "#7a63b0"], [1, "#3f007d"]]
COLOR_OPTIONS = {   # values -> colour, colour range, legend ticks (values, labels), legend text
    "score": dict(val=np.log10(fac["score"]), lim=(0, np.log10(max(6, fac["score"].max()))),
                  ticks=(np.log10([1, 2, 3, 5]), ["1", "2", "3", "5"]),
                  text=f"median canonicity score of the {UNIT_LABEL} (log scale)"),
    "listed": dict(val=fac["listed"], lim=(0.55, 1), ticks=([0.6, 0.7, 0.8, 0.9, 1], ["60%", "70%", "80%", "90%", "100%"]),
                   text=f"share of {UNIT_LABEL} in the canon list"),
    "median_rank": dict(val=-np.log10(fac["median_rank"]), lim=(-np.log10(5000), -np.log10(200)),
                        ticks=(-np.log10([3000, 1000, 500, 300]), ["3,000", "1,000", "500", "300"]),
                        text=f"median rank of the {UNIT_LABEL} in the canon list (log scale)"),
}
COL = COLOR_OPTIONS.get(COLOR14)
Y_MAX = np.ceil(fac["top"].max() * 10) / 10 + 0.02   # a little above the highest tick

fig = make_subplots(rows=3, cols=1, vertical_spacing=0.085,
                    subplot_titles=[f"<b>{t}</b>  <span style='color:{INK2}'>· {s}</span>" for _, t, s in PANELS])
for r, (xc, _, _) in enumerate(PANELS, start=1):
    d_ = fac.dropna(subset=[xc])
    fig.add_scatter(x=d_[xc], y=d_["top"], mode="markers+text", text=d_.index,
                    textposition=[POS14[xc].get(n, "middle right") for n in d_.index],
                    textfont=dict(size=FS - 1, color=INK),
                    marker=dict(size=15, color=COL["val"].loc[d_.index] if COL else DOT_COLOR, colorscale=DOT_SCALE,
                                cmin=COL["lim"][0] if COL else None, cmax=COL["lim"][1] if COL else None,
                                line=dict(color=SURFACE, width=2), showscale=bool(COL) and r == 1,
                                colorbar=dict(orientation="h", x=0.585, xanchor="left", y=0, yref="container", yanchor="bottom", ypad=20,
                                              len=240, lenmode="pixels", thickness=14, outlinewidth=0, ticks="",
                                              tickvals=COL["ticks"][0] if COL else None, ticktext=COL["ticks"][1] if COL else None, tickfont=dict(size=FS - 2))),
                    customdata=np.c_[d_["size"], d_["year"], d_["gutenberg"].fillna(-1), d_["score"], d_["listed"]], showlegend=False,
                    hovertemplate="<b>%{text}</b><br>%{y:.0%} of the " + UNIT_LABEL + " in the top " + str(TOP)
                                  + "<br>%{customdata[0]:,} " + UNIT_LABEL + ", median year %{customdata[1]:.0f}"
                                  + "<br>median canonicity score %{customdata[3]:.1f}, %{customdata[4]:.0%} in the canon list<extra></extra>",
                    row=r, col=1)

# panel headings: left-aligned above each panel
for a in fig.layout.annotations[:len(PANELS)]:
    a.update(x=0, xanchor="left", font=dict(size=FS + 3, color=INK))
# colour legend title, left of the colour bar (legend row below the figure)
fig.add_annotation(text=f"<b>Colour</b>  <span style='color:{INK2}'>· {COL['text'] if COL else ''}</span>",
                   xref="paper", yref="paper", x=0, y=1, xanchor="left", yanchor="top", align="left",
                   showarrow=False, font=dict(size=FS, color=INK))
# corpora missing in a panel: right-aligned in the heading line of that panel
for (xc, _, _), head in zip(PANELS, fig.layout.annotations[:len(PANELS)]):
    missing = [n for n in ORDER if pd.isna(fac.loc[n, xc])]
    if missing:
        fig.add_annotation(text=f"not shown (no source per text):<br>{', '.join(missing)}", align="right",
                           xref="paper", yref="paper", x=1, y=head.y, xanchor="right", yanchor="bottom",
                           showarrow=False, font=dict(size=FS - 3, color=INK2))

lo, hi = fac["size"].min(), fac["size"].max()
fig.update_xaxes(type="log", tickvals=[50, 100, 200, 500, 1000, 2000, 5000],
                 range=[np.log10(lo / 1.4), np.log10(hi * 1.6)], row=ROW["size"], col=1)
fig.update_xaxes(range=[1785, 1920], dtick=20, row=ROW["year"], col=1)
fig.update_xaxes(range=[-0.06, 1.1], tickvals=[0, 0.25, 0.5, 0.75, 1], tickformat=".0%", row=ROW["gutenberg"], col=1)
fig.update_xaxes(showgrid=True, gridcolor="#f1f0ec", zeroline=False, tickfont=dict(size=FS - 1))
fig.update_yaxes(tickformat=".0%", range=[0, Y_MAX], dtick=0.1, tickfont=dict(size=FS - 1), zeroline=False)
fig.update_yaxes(title=dict(text=f"share of {UNIT_LABEL} in the top {TOP}", font=dict(size=FS + 1)))   # every panel
style(fig, "Three factors", "each dot is a corpus; same y-axis in all panels", height=1470, legend=False, font_size=FS)
fig.update_layout(width=1000, margin=dict(t=(100 if SHOW_TITLES else 50), l=20, r=30, b=135))
mg = fig.layout.margin
leg = [a for a in fig.layout.annotations if (a.text or "").startswith("<b>Colour")][0]
if not COL:   # one colour: no colour legend, no extra space below
    leg.visible = False
    fig.update_layout(margin=dict(b=40), height=fig.layout.height - mg.b + 40)
    mg = fig.layout.margin
leg.update(yanchor="bottom", y=-(mg.b - 32) / (fig.layout.height - mg.t - mg.b))   # level with the colour bar

# HTML with linked highlighting: hovering a corpus highlights it in all panels
LINK_JS = """
var gd = document.getElementById('{plot_id}');
var idx = [], names = [];
gd.data.forEach(function(t, i) { if (t.text && t.mode && t.mode.indexOf('markers') > -1) { idx.push(i); names.push(t.text); } });
function paint(sel) {
  var op = [], ls = [], sz = [], tc = [];
  names.forEach(function(T) {
    op.push(T.map(function(n) { return (sel === null || n === sel) ? 1 : 0.2; }));
    ls.push(T.map(function(n) { return n === sel ? '#0b0b0b' : '#ffffff'; }));
    sz.push(T.map(function(n) { return n === sel ? 20 : 15; }));
    tc.push(T.map(function(n) { return (sel === null || n === sel) ? '#0b0b0b' : 'rgba(82,81,78,0.2)'; }));
  });
  Plotly.restyle(gd, {'marker.opacity': op, 'marker.line.color': ls, 'marker.size': sz, 'textfont.color': tc}, idx);
}
gd.on('plotly_hover', function(ev) { paint(ev.points[0].text); });
gd.on('plotly_unhover', function() { paint(null); });
"""
show(fig, f"fig14_three_factors_{UNIT14}", post_script=LINK_JS)

# Fig. 15: Non-male authors and canonicity

One dot per corpus. x = share of works by **non-male authors**, y = as in Fig. 14 (share of works in the top 200, see `UNIT14`),
colour = corpus size (number of works, log scale) or source (share of texts from Gutenberg; open circles = no source per text, as in Fig. 14).
Both versions are saved (`fig15_gender_…_by_size`, `…_by_source`).

- Gender from the GND (`GND_gender`; `gnd_all_mit_namensvarianten.csv` + `gnd_add.csv`). The GND only knows *männlich*, *weiblich*
  and *unbekannt*, so in practice "non-male" = female.
- A work with several authors counts as non-male if at least one of them is not male.
- Works with unknown gender (GND *unbekannt*, or no GND ID: anonymous, placeholders) are left out of the share.
  This concerns mainly 19C prose (about 15 % of its works); all other corpora at most 3 %.

In [7]:
g1 = pd.read_csv(GND_FILES[0], sep=";", dtype={"gndIdentifier": str}, usecols=["gndIdentifier", "GND_gender"])
g2 = pd.read_csv(GND_FILES[1], dtype=str).rename(columns={"GND": "gndIdentifier"})[["gndIdentifier", "GND_gender"]]
gender = (pd.concat([g1, g2]).assign(gndIdentifier=lambda d: d["gndIdentifier"].str.strip())
          .drop_duplicates("gndIdentifier").set_index("gndIdentifier")["GND_gender"])

def work_gender(g):
    if g.str.contains("Weiblich", na=False).any():
        return "non-male"
    return "male" if g.eq("Männlich").all() else "unknown"

g_work = wa.assign(g=wa["gndIdentifier"].map(gender)).groupby("work_id")["g"].agg(work_gender)
data["gender"] = data["work_id"].map(g_work).fillna("unknown")     # works without GND ID -> unknown
fac["nonmale"] = data[data["gender"].ne("unknown")].groupby("name")["gender"].apply(lambda s: s.eq("non-male").mean())
fac["gender_known"] = data.groupby("name")["gender"].apply(lambda s: s.ne("unknown").mean())

POS15 = {"Bildungsromane": "top center", "d-RoRo": "middle left", "d-prose": "middle left",
         "Neuer Novellenschatz": "middle left"}   # label positions where the default ("middle right") would collide
NO_DATA = "#8a8984"   # outline of the open circles = no value for the colour measure
COLOR15 = {   # one figure per colouring: values, colour scale, legend ticks and text
    "size": dict(val=np.log10(fac["size"]), scale=[[0, "#8fc1e3"], [0.5, "#3a7fc1"], [1, "#08306b"]],   # light -> dark blue
                 lim=(np.log10(fac["size"].min()), np.log10(fac["size"].max())),
                 ticks=(np.log10([100, 300, 1000, 3000]), ["100", "300", "1,000", "3,000"]),
                 text=f"corpus size, number of {UNIT_LABEL} (log scale)"),
    "source": dict(val=fac["gutenberg"], scale=[[0, "#8fc1e3"], [0.5, "#3a7fc1"], [1, "#08306b"]],     # light -> dark blue
                   lim=(0, 1), ticks=([0, 0.25, 0.5, 0.75, 1], ["0%", "25%", "50%", "75%", "100%"]),
                   text="share of texts from Project Gutenberg"),
}

def fig15(by):
    cfg = COLOR15[by]
    has = cfg["val"].notna()
    fig = go.Figure()
    for sel, colored in ((has, True), (~has, False)):   # coloured dots + grey dots without a value
        d_ = fac[sel]
        if d_.empty:
            continue
        marker = dict(size=15, line=dict(color=SURFACE, width=2))
        if colored:
            marker.update(color=cfg["val"][sel], colorscale=cfg["scale"], cmin=cfg["lim"][0], cmax=cfg["lim"][1], showscale=True,
                          colorbar=dict(orientation="h", x=0.62, xanchor="left", y=0, yref="container", yanchor="bottom", ypad=20,
                                        len=260, lenmode="pixels", thickness=14, outlinewidth=0, ticks="",
                                        tickvals=cfg["ticks"][0], ticktext=cfg["ticks"][1], tickfont=dict(size=FS - 2)))
        else:
            marker.update(color=SURFACE, size=13, line=dict(color=NO_DATA, width=2))   # open circle
        fig.add_scatter(x=d_["nonmale"], y=d_["top"], mode="markers+text", text=d_.index, showlegend=False,
                        textposition=[POS15.get(n, "middle right") for n in d_.index], textfont=dict(size=FS - 1, color=INK),
                        marker=marker, customdata=np.c_[d_["gender_known"], d_["size"], d_["gutenberg"].fillna(-1)],
                        hovertemplate="<b>%{text}</b><br>%{x:.0%} of the works by non-male authors"
                                      "<br>%{y:.0%} of the " + UNIT_LABEL + " in the top " + str(TOP)
                                      + "<br>gender known for %{customdata[0]:.0%} of the works"
                                      + "<br>%{customdata[1]:,} " + UNIT_LABEL + "<extra></extra>")
    fig.add_annotation(text=f"<b>Gender</b>  <span style='color:{INK2}'>· share of works by non-male authors</span>",
                       xref="paper", yref="paper", x=0, y=1, xanchor="left", yanchor="bottom", yshift=8,
                       showarrow=False, font=dict(size=FS + 3, color=INK))
    fig.update_xaxes(range=[0, np.ceil(fac["nonmale"].max() * 10) / 10 + 0.02], dtick=0.1, tickformat=".0%",
                     showgrid=True, gridcolor="#f1f0ec", zeroline=False, tickfont=dict(size=FS - 1))
    fig.update_yaxes(range=[0, Y_MAX], dtick=0.1, tickformat=".0%", zeroline=False, tickfont=dict(size=FS - 1),
                     title=dict(text=f"share of {UNIT_LABEL} in the top {TOP}", font=dict(size=FS + 1)))
    style(fig, "Gender", None, height=710, legend=False, font_size=FS)
    fig.update_layout(width=1000, margin=dict(t=60, l=20, r=30, b=130))
    # colour legend row below the figure: title left of the colour bar, note on grey dots below it
    mg = fig.layout.margin
    y_leg = -(mg.b - 32) / (fig.layout.height - mg.t - mg.b)
    note = (f"<br><span style='font-size:{FS - 3}px'>open circles: no source per text<br>({', '.join(fac.index[~has])})</span>"
            if (~has).any() else "")
    fig.add_annotation(text=f"<b>Colour</b>  <span style='color:{INK2}'>· {cfg['text']}{note}</span>",
                       xref="paper", yref="paper", x=0, y=y_leg, xanchor="left", yanchor="bottom" if not note else "middle",
                       yshift=0 if not note else 8, align="left", showarrow=False, font=dict(size=FS, color=INK))
    return fig

for by in COLOR15:
    show(fig15(by), f"fig15_gender_{UNIT14}_by_{by}")
fac[["nonmale", "gender_known", "top"]].sort_values("nonmale").style.format("{:.0%}")

,nonmale,gender_known,top
name,,,
DTA fiction,7%,96%,43%
CGLF,11%,99%,29%
d-prose,13%,99%,24%
Novellenschatz,14%,100%,24%
de-Corp,15%,97%,20%
Neuer Novellenschatz,21%,100%,14%
glance,23%,98%,16%
19C prose,23%,85%,25%
Bildungsromane,33%,99%,47%
